In [11]:
from pathlib import Path
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

folder = Path("hw1")

image_paths = []

for ext in ["*.png", "*.jpg", "*.jpeg"]:
    image_paths.extend(folder.glob(ext))

image_paths = sorted(image_paths)

In [12]:
def classify_image(image_path):
    img_bgr = cv.imread(str(image_path))

    if img_bgr is None:
        return None

    img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
    img_hsv = cv.cvtColor(img_bgr, cv.COLOR_BGR2HSV)

    H = img_hsv[:, :, 0]
    S = img_hsv[:, :, 1]
    V = img_hsv[:, :, 2]
    height, width = H.shape

    red_mask = (((H >= 0) & (H <= 15)) | ((H >= 170) & (H <= 179))) & (S > 90) & (V > 120)
    yellow_mask = (H >= 15) & (H <= 35) & (S > 90) & (V > 120)
    green_mask = (H >= 35) & (H <= 90) & (S > 80) & (V > 100)

    x1 = int(width * 0.15)
    x2 = int(width * 0.85)

    red_area = red_mask[int(height * 0.05):int(height * 0.45), x1:x2]
    yellow_area = yellow_mask[int(height * 0.25):int(height * 0.70), x1:x2]
    green_area = green_mask[int(height * 0.45):int(height * 0.95), x1:x2]

    red_rows = np.sum(red_area, axis=1)
    yellow_rows = np.sum(yellow_area, axis=1)
    green_rows = np.sum(green_area, axis=1)

    red_score = np.max(red_rows)
    yellow_score = np.max(yellow_rows)
    green_score = np.max(green_rows)

    scores = {
        "red": red_score,
        "yellow": yellow_score,
        "green": green_score
    }
    prediction = max(scores, key=scores.get)

    masks = {
        "red": red_mask,
        "yellow": yellow_mask,
        "green": green_mask
    }
    selected_mask = masks[prediction]

    detected_img = np.copy(img_rgb)
    detected_img[np.logical_not(selected_mask)] = 0

    return {
        "img_rgb": img_rgb,
        "prediction": prediction,
        "selected_mask": selected_mask,
        "detected_img": detected_img,

        "scores": {
            "red": int(red_score),
            "yellow": int(yellow_score),
            "green": int(green_score)
        }
    }

In [13]:
results = []

for image_path in image_paths:
    result = classify_image(image_path)
    if result is None:
        continue

    results.append({"filename": image_path.name, "class": result["prediction"]})

    # fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # axes[0].imshow(result["img_rgb"])
    # axes[0].set_title("Original")

    # axes[1].imshow(result["selected_mask"], cmap="gray")
    # axes[1].set_title("Selected mask")

    # axes[2].imshow(result["detected_img"])
    # axes[2].set_title(f'Detected: {result["prediction"].upper()}')

    # for ax in axes:
    #     ax.axis("off")

    # plt.suptitle(image_path.name)
    # plt.tight_layout()
    # plt.show()


files_by_class = {
    "red": [],
    "yellow": [],
    "green": []
}

for result in results:
    files_by_class[result["class"]].append(result["filename"])


for color, filenames in files_by_class.items():
    with open(f"{color}.txt", "w", encoding="utf-8") as file:
        for filename in filenames:
            file.write(filename + "\n")